# LeetCode 780 — Reaching Points bằng Python

- Link: https://leetcode.com/problems/reaching-points/
- Độ khó: **Hard**
- Chủ đề: Toán, Đi ngược (work backwards), Thuật toán Euclid, Phép chia lấy dư

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def reachingPoints(self, sx: int, sy: int, tx: int, ty: int) -> bool:
```

## 1. Giới thiệu & mục tiêu học

Từ điểm `(x, y)` ta được biến thành `(x + y, y)` hoặc `(x, x + y)`. Hỏi từ `(sx, sy)` có tới được `(tx, ty)` không? Các số lên tới `10⁹`, nên mọi cách "đi tới" đều bùng nổ. Mẹo then chốt: **đi ngược** từ đích — khi đó mỗi bước **chỉ có một lựa chọn**, và chuỗi phép trừ lặp lại chính là **thuật toán Euclid** → thu gọn bằng phép `%`.

Sau notebook này bạn sẽ:

1. Thấy vì sao đi xuôi tạo cây nhị phân khổng lồ còn đi ngược chỉ là một đường thẳng.
2. Nhận ra "trừ lặp nhiều lần" = phép **chia lấy dư** (Euclid).
3. Xử lý chính xác trường hợp biên khi một toạ độ đã chạm giá trị xuất phát.
4. Cài 4 lời giải: BFS xuôi, đi ngược bằng phép trừ, đi ngược bằng `%` (vòng lặp và đệ quy).
5. Chứng minh tính đúng và phân tích độ phức tạp `O(log max(tx, ty))`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho bốn số nguyên dương `sx, sy, tx, ty`. Một **phép biến đổi** đưa điểm `(x, y)` thành `(x, x + y)` hoặc `(x + y, y)`. Trả về `True` nếu có thể biến `(sx, sy)` thành `(tx, ty)` bằng một dãy (có thể rỗng) các phép biến đổi, ngược lại `False`.

**Ví dụ 1**

```text
Input: sx = 1, sy = 1, tx = 3, ty = 5
Output: true
Giải thích: (1, 1) -> (1, 2) -> (3, 2) -> (3, 5)
```

**Ví dụ 2**

```text
Input: sx = 1, sy = 1, tx = 2, ty = 2
Output: false
```

**Ví dụ 3**

```text
Input: sx = 1, sy = 1, tx = 1, ty = 1
Output: true
```

**Ràng buộc**

- `1 <= sx, sy, tx, ty <= 10⁹`

In [1]:
EXAMPLES = [
    ((1, 1, 3, 5), True),
    ((1, 1, 2, 2), False),
    ((1, 1, 1, 1), True),
    ((9, 5, 12, 8), False),
    ((3, 7, 3, 4), False),
    ((1, 1, 1000000000, 1), True),
]
for args, exp in EXAMPLES:
    print(args, "->", exp)

(1, 1, 3, 5) -> True
(1, 1, 2, 2) -> False
(1, 1, 1, 1) -> True
(9, 5, 12, 8) -> False
(3, 7, 3, 4) -> False
(1, 1, 1000000000, 1) -> True


## 3. Kiến thức nền

### 3.1. Phép chia lấy dư `%` và chia nguyên `//`

Với `a ≥ 0, b > 0`: `a = (a // b) * b + a % b`, và `0 ≤ a % b < b`. Trừ `b` khỏi `a` lặp lại `k` lần cho tới khi `a < b` chính là `a % b` (với `k = a // b`).

In [2]:
a, b = 23, 5
x = a
k = 0
while x >= b:
    x -= b
    k += 1
print("trừ lặp:", x, "sau", k, "lần |", "a % b =", a % b, "| a // b =", a // b)

trừ lặp: 3 sau 4 lần | a % b = 3 | a // b = 4


### 3.2. Thuật toán Euclid

`gcd(a, b) = gcd(b, a % b)`, dừng khi `b = 0`. Mỗi 2 bước, số lớn ít nhất giảm một nửa → `O(log max(a, b))` bước. Đi ngược trong bài này có cấu trúc **giống hệt** Euclid.

In [3]:
def gcd_trace(a, b):
    steps = 0
    while b:
        print(f"gcd({a}, {b})")
        a, b = b, a % b
        steps += 1
    print("=>", a, "sau", steps, "bước")
    return a

gcd_trace(1071, 462)

gcd(1071, 462)
gcd(462, 147)
gcd(147, 21)
=> 21 sau 3 bước


21

### 3.3. BFS / DFS trên đồ thị ẩn

Không cần dựng đồ thị: từ mỗi trạng thái tự sinh các trạng thái kế tiếp. Ở đây trạng thái là cặp `(x, y)`; ta cắt nhánh khi vượt quá đích (vì giá trị chỉ tăng).

In [4]:
from collections import deque

def successors(x, y):
    return [(x + y, y), (x, x + y)]

print(successors(1, 1), successors(1, 2))

[(2, 1), (1, 2)] [(3, 2), (1, 3)]


### 3.4. Đệ quy và giới hạn độ sâu

Python mặc định giới hạn độ sâu đệ quy ~1000. Một lời giải đệ quy chỉ an toàn khi độ sâu nhỏ (ví dụ `O(log n)`). Đi ngược bằng phép trừ có thể sâu tới `10⁹` → bắt buộc dùng `%`.

In [5]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đi xuôi: cây nhị phân

Từ `(1, 1)`:

```text
                 (1,1)
            /            \
        (2,1)            (1,2)
       /     \          /     \
   (3,1)   (2,3)    (3,2)   (1,3)
   ...
```

Mỗi mức nhân đôi số nút → không thể duyệt hết khi đích cỡ `10⁹`.

### 4.2. Đi ngược: chỉ một lựa chọn

Giả sử đang ở `(tx, ty)` với `tx > ty`. Bước cuối cùng là gì?

- Nếu bước cuối là `(x, x + y) → (tx, ty)` thì `ty = x + y > x = tx` — mâu thuẫn với `tx > ty`.
- Vậy bước cuối phải là `(x + y, y)`: tức điểm trước là `(tx - ty, ty)`.

Tương tự, nếu `ty > tx` thì điểm trước là `(tx, ty - tx)`. Nếu `tx == ty` (và cả hai ≥ 1) thì **không có** điểm trước hợp lệ (điểm trước sẽ có toạ độ 0).

→ Đi ngược là một **đường thẳng duy nhất** về phía gốc!

### 4.3. Tính tay ví dụ 1: `(1,1) → (3,5)`

```text
(3, 5): ty > tx → (3, 2)
(3, 2): tx > ty → (1, 2)
(1, 2): ty > tx → (1, 1)  = (sx, sy)  ✔
```

### 4.4. Tính tay `(1, 1) → (2, 2)`

`(2, 2)`: `tx == ty` → không có điểm trước; `(2,2) ≠ (1,1)` → `False`.

### 4.5. Trừ lặp nhiều lần

`(1, 10⁹) ← ... ` từ `(1, 1)`: đi ngược `(1, 10⁹) → (1, 10⁹ - 1) → ... → (1, 1)` mất `10⁹ - 1` bước. Đó là lý do cần `%`.

In [6]:
def backward_path(tx, ty, limit=20):
    path = [(tx, ty)]
    while tx != ty and len(path) < limit:
        if tx > ty:
            tx -= ty
        else:
            ty -= tx
        path.append((tx, ty))
    return path

print(backward_path(3, 5))
print(backward_path(2, 2))
print(backward_path(19, 7))

[(3, 5), (3, 2), (1, 2), (1, 1)]
[(2, 2)]
[(19, 7), (12, 7), (5, 7), (5, 2), (3, 2), (1, 2), (1, 1)]


## 5. Cách 1 — Brute Force: BFS xuôi có cắt nhánh

**Ý tưởng.** Từ `(sx, sy)`, BFS sinh các điểm `(x+y, y)` và `(x, x+y)`, bỏ qua điểm có `x > tx` hoặc `y > ty` (vì toạ độ chỉ tăng). Nếu gặp `(tx, ty)` → `True`.

Chỉ dùng được khi `tx, ty` nhỏ (ví dụ ≤ 60) — dùng làm **chuẩn đối chiếu**.

In [7]:
def reach_bfs(sx, sy, tx, ty):
    seen = {(sx, sy)}
    q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        if (x, y) == (tx, ty):
            return True
        for nx, ny in successors(x, y):
            if nx <= tx and ny <= ty and (nx, ny) not in seen:
                seen.add((nx, ny))
                q.append((nx, ny))
    return False

for args, exp in EXAMPLES[:5]:
    print(args, reach_bfs(*args), "(kỳ vọng", exp, ")")
    assert reach_bfs(*args) == exp

(1, 1, 3, 5) True (kỳ vọng True )
(1, 1, 2, 2) False (kỳ vọng False )
(1, 1, 1, 1) True (kỳ vọng True )
(9, 5, 12, 8) False (kỳ vọng False )
(3, 7, 3, 4) False (kỳ vọng False )


**Độ phức tạp.** Số điểm `(x, y)` với `x ≤ tx, y ≤ ty` tối đa `tx · ty` → **`O(tx · ty)`** thời gian và bộ nhớ (thực tế ít hơn nhiều nhờ chỉ thăm điểm tới được, nhưng vẫn rất lớn). Với `10⁹` hoàn toàn bất khả thi.

**Nút thắt.** Ta khám phá một cây phân nhánh, trong khi đi **ngược** thì không phân nhánh.

## 6. Bước đột phá

**Insight 1 — Đi ngược là duy nhất.**
- *Nếu* `tx > ty` *thì* điểm trước là `(tx - ty, ty)`.
- *Nếu* `ty > tx` *thì* điểm trước là `(tx, ty - tx)`.
- *Nếu* `tx == ty` *thì* không đi ngược được nữa.

**Insight 2 — Gộp phép trừ lặp thành `%`.** Khi `tx > ty`, ta trừ `ty` liên tục cho tới khi `tx ≤ ty`... nhưng phải **dừng đúng lúc** nếu toạ độ `y` đã bằng `sy`, vì từ đó trở đi chỉ còn `tx` giảm dần theo bước `ty`:

- *Nếu* `tx > ty` *và* `ty > sy` *thì* `tx %= ty` (an toàn: ta còn phải giảm `ty` sau này, mà muốn giảm `ty` thì cần `tx < ty`).
- *Nếu* `tx > ty` *và* `ty == sy` *thì* chỉ còn cách trừ `ty` khỏi `tx` cho tới `sx`: kiểm tra `tx ≥ sx` và `(tx - sx) % ty == 0`.
- Đối xứng cho trường hợp `ty > tx`.

**Invariant:** tại mọi thời điểm, `(sx, sy)` tới được `(tx, ty)` ban đầu ⇔ `(sx, sy)` tới được `(tx, ty)` hiện tại (vì mỗi bước lùi là bước lùi *bắt buộc*).

**Điều kiện dừng:** khi `tx < sx` hoặc `ty < sy` → không thể (toạ độ chỉ tăng khi đi xuôi).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đi ngược bằng phép trừ

Đúng về logic, nhưng `O(max(tx, ty))` bước trong trường hợp xấu (ví dụ `(1, 10⁹)`) → TLE. Vẫn hữu ích để hiểu và đối chiếu.

In [8]:
def reach_subtract(sx, sy, tx, ty, max_steps=10**7):
    steps = 0
    while tx >= sx and ty >= sy:
        if (tx, ty) == (sx, sy):
            return True
        if tx == ty:
            return False
        if tx > ty:
            tx -= ty
        else:
            ty -= tx
        steps += 1
        if steps > max_steps:
            raise RuntimeError("quá nhiều bước — sẽ TLE")
    return False

for args, exp in EXAMPLES[:5]:
    assert reach_subtract(*args) == exp
print("Đi ngược bằng phép trừ: OK trên ví dụ nhỏ")
try:
    reach_subtract(1, 1, 10**9, 1, max_steps=10**5)
except RuntimeError as e:
    print("(1,1)->(1e9,1):", e)

Đi ngược bằng phép trừ: OK trên ví dụ nhỏ
(1,1)->(1e9,1): quá nhiều bước — sẽ TLE


Code nộp LeetCode (sẽ TLE với test lớn, để tham khảo):

In [9]:
class SolutionSubtract:
    def reachingPoints(self, sx: int, sy: int, tx: int, ty: int) -> bool:
        while tx >= sx and ty >= sy:
            if tx == sx and ty == sy:
                return True
            if tx == ty:
                return False
            if tx > ty:
                tx -= ty
            else:
                ty -= tx
        return False

for args, exp in EXAMPLES[:5]:
    assert SolutionSubtract().reachingPoints(*args) == exp
print("SolutionSubtract: OK (ví dụ nhỏ)")

SolutionSubtract: OK (ví dụ nhỏ)


**Độ phức tạp cách 2:** `O(max(tx, ty) / min(...))` bước, xấu nhất `O(10⁹)`. Bộ nhớ `O(1)`.

### 7.2. Cách 3 — Đi ngược bằng `%` (lời giải chuẩn)

```text
while tx >= sx and ty >= sy:
    if tx == ty: break
    if tx > ty:
        if ty > sy: tx %= ty
        else: return (tx - sx) % ty == 0      # ty == sy
    else:
        if tx > sx: ty %= tx
        else: return (ty - sy) % tx == 0      # tx == sx
return tx == sx and ty == sy
```

Giải thích từng nhánh:

- `tx == ty`: không lùi được nữa → thoát vòng, so sánh với `(sx, sy)`.
- `tx > ty, ty > sy`: ta cần giảm `ty` về sau; chỉ giảm `ty` khi `tx < ty`; vậy phải lùi `tx` hết mức (về `tx % ty`). Nếu `tx % ty < sx` thì vòng lặp kết thúc và trả `False` — đúng, vì không có cách nào tốt hơn.
- `tx > ty, ty == sy`: `ty` đã đúng, không được giảm nữa. Chỉ còn trừ `ty` khỏi `tx`: tới được `sx` ⇔ `tx - sx` là bội của `ty` (vòng `while` đảm bảo `tx ≥ sx`).

In [10]:
def reach_mod(sx, sy, tx, ty):
    while tx >= sx and ty >= sy:
        if tx == ty:
            break
        if tx > ty:
            if ty > sy:
                tx %= ty
            else:
                return (tx - sx) % ty == 0
        else:
            if tx > sx:
                ty %= tx
            else:
                return (ty - sy) % tx == 0
    return tx == sx and ty == sy

for args, exp in EXAMPLES:
    assert reach_mod(*args) == exp, args
print("Đi ngược bằng %: OK")

Đi ngược bằng %: OK


Code nộp LeetCode cho cách 3:

In [11]:
class SolutionMod:
    def reachingPoints(self, sx: int, sy: int, tx: int, ty: int) -> bool:
        while tx >= sx and ty >= sy:
            if tx == ty:
                break
            if tx > ty:
                if ty > sy:
                    tx %= ty
                else:
                    return (tx - sx) % ty == 0
            else:
                if tx > sx:
                    ty %= tx
                else:
                    return (ty - sy) % tx == 0
        return tx == sx and ty == sy

for args, exp in EXAMPLES:
    assert SolutionMod().reachingPoints(*args) == exp
print("SolutionMod: OK")

SolutionMod: OK


**Độ phức tạp cách 3:** giống Euclid → **`O(log max(tx, ty))`** thời gian, `O(1)` bộ nhớ.

### 7.3. Cách 4 — Đệ quy kiểu "nhảy nhiều bước một lúc"

Một cách viết khác cũng hay gặp: khi `tx > ty`, lùi `tx` về giá trị nhỏ nhất **không nhỏ hơn `sx`** mà vẫn đồng dư với `tx` theo `ty`:

```text
k = (tx - sx) // ty        # số lần có thể trừ mà vẫn ≥ sx
tx -= max(k, 1) * ty  ...
```

Cách viết dễ nhớ hơn: lùi tối đa `k = (tx - sx) // ty` lần (nếu `k = 0` thì không lùi được → `False` trừ khi đã bằng). Độ sâu đệ quy `O(log)` nên an toàn.

In [12]:
def reach_rec(sx, sy, tx, ty):
    if tx < sx or ty < sy:
        return False
    if tx == sx and ty == sy:
        return True
    if tx == ty:
        return False
    if tx > ty:
        k = (tx - sx) // ty          # số lần trừ ty tối đa mà tx vẫn >= sx
        if k == 0:
            return False
        return reach_rec(sx, sy, tx - k * ty, ty)
    else:
        k = (ty - sy) // tx
        if k == 0:
            return False
        return reach_rec(sx, sy, tx, ty - k * tx)

for args, exp in EXAMPLES:
    assert reach_rec(*args) == exp, args
print("Đệ quy nhảy nhiều bước: OK")

Đệ quy nhảy nhiều bước: OK


Vì sao trừ `k = (tx - sx) // ty` lần là an toàn? Sau khi trừ, `sx ≤ tx' < sx + ty`. Nếu `tx'` vẫn `> ty` thì lần lùi tiếp theo lại phải trừ `ty` → `tx'' < sx` → thất bại. Vậy hoặc `tx' == sx` (rồi xét `ty`), hoặc `tx' < ty` (chuyển sang giảm `ty`). Mọi khả năng hợp lệ đều được giữ lại. Mỗi lần gọi đệ quy luân phiên giảm `tx` và `ty` như Euclid → độ sâu `O(log)`.

In [13]:
class SolutionRec:
    def reachingPoints(self, sx: int, sy: int, tx: int, ty: int) -> bool:
        if tx < sx or ty < sy:
            return False
        if tx == sx and ty == sy:
            return True
        if tx == ty:
            return False
        if tx > ty:
            k = (tx - sx) // ty
            return k > 0 and self.reachingPoints(sx, sy, tx - k * ty, ty)
        k = (ty - sy) // tx
        return k > 0 and self.reachingPoints(sx, sy, tx, ty - k * tx)

for args, exp in EXAMPLES:
    assert SolutionRec().reachingPoints(*args) == exp
print("SolutionRec: OK")

SolutionRec: OK


## 8. TRACE từng bước

Trace cách 3 trên vài bộ input: in `(tx, ty)` và nhánh được chọn ở mỗi vòng.

In [14]:
def trace_mod(sx, sy, tx, ty):
    print(f"start=({sx},{sy}) target=({tx},{ty})")
    step = 0
    while tx >= sx and ty >= sy:
        if tx == ty:
            print(f"  vòng {step}: tx == ty = {tx} -> dừng")
            break
        if tx > ty:
            if ty > sy:
                print(f"  vòng {step}: ({tx},{ty}) tx>ty, ty>sy -> tx %= ty -> ({tx % ty},{ty})")
                tx %= ty
            else:
                ok = (tx - sx) % ty == 0
                print(f"  vòng {step}: ({tx},{ty}) tx>ty, ty==sy -> (tx-sx)%ty = {(tx - sx) % ty} -> {ok}")
                return ok
        else:
            if tx > sx:
                print(f"  vòng {step}: ({tx},{ty}) ty>tx, tx>sx -> ty %= tx -> ({tx},{ty % tx})")
                ty %= tx
            else:
                ok = (ty - sy) % tx == 0
                print(f"  vòng {step}: ({tx},{ty}) ty>tx, tx==sx -> (ty-sy)%tx = {(ty - sy) % tx} -> {ok}")
                return ok
        step += 1
    res = tx == sx and ty == sy
    print(f"  kết thúc ở ({tx},{ty}) -> {res}")
    return res

trace_mod(1, 1, 3, 5)
print()
trace_mod(2, 3, 11, 30)
print()
trace_mod(1, 1, 2, 2)
print()
trace_mod(3, 7, 3, 4)

start=(1,1) target=(3,5)
  vòng 0: (3,5) ty>tx, tx>sx -> ty %= tx -> (3,2)
  vòng 1: (3,2) tx>ty, ty>sy -> tx %= ty -> (1,2)
  vòng 2: (1,2) ty>tx, tx==sx -> (ty-sy)%tx = 0 -> True

start=(2,3) target=(11,30)
  vòng 0: (11,30) ty>tx, tx>sx -> ty %= tx -> (11,8)
  vòng 1: (11,8) tx>ty, ty>sy -> tx %= ty -> (3,8)
  vòng 2: (3,8) ty>tx, tx>sx -> ty %= tx -> (3,2)
  kết thúc ở (3,2) -> False

start=(1,1) target=(2,2)
  vòng 0: tx == ty = 2 -> dừng
  kết thúc ở (2,2) -> False

start=(3,7) target=(3,4)
  kết thúc ở (3,4) -> False


False

Bảng trạng thái cho `(2,3) → (11,30)`:

| vòng | (tx, ty) | điều kiện | hành động | kết quả |
|---|---|---|---|---|
| 0 | (11, 30) | ty > tx, tx = 11 > sx = 2 | `ty %= 11` | (11, 8) |
| 1 | (11, 8) | tx > ty, ty = 8 > sy = 3 | `tx %= 8` | (3, 8) |
| 2 | (3, 8) | ty > tx, tx = 3 > sx = 2 | `ty %= 3` | (3, 2) |
| — | (3, 2) | ty = 2 < sy = 3 | thoát vòng | `False` |

Kiểm chứng bằng BFS:

In [15]:
print(reach_bfs(2, 3, 11, 30), reach_mod(2, 3, 11, 30))
# Một ví dụ True dài hơn: (2,3) -> (2,5) -> (7,5) -> (12,5) -> (12,17) -> (12,29)
print(reach_bfs(2, 3, 12, 29), reach_mod(2, 3, 12, 29))
trace_mod(2, 3, 12, 29)

False False
True True
start=(2,3) target=(12,29)
  vòng 0: (12,29) ty>tx, tx>sx -> ty %= tx -> (12,5)
  vòng 1: (12,5) tx>ty, ty>sy -> tx %= ty -> (2,5)
  vòng 2: (2,5) ty>tx, tx==sx -> (ty-sy)%tx = 0 -> True


True

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def reachingPoints(self, sx: int, sy: int, tx: int, ty: int) -> bool:
        while tx >= sx and ty >= sy:
            if tx == ty:
                break
            if tx > ty:
                if ty > sy:
                    tx %= ty
                else:                       # ty == sy: chỉ còn giảm tx theo bước ty
                    return (tx - sx) % ty == 0
            else:
                if tx > sx:
                    ty %= tx
                else:                       # tx == sx: chỉ còn giảm ty theo bước tx
                    return (ty - sy) % tx == 0
        return tx == sx and ty == sy

for args, exp in EXAMPLES:
    assert Solution().reachingPoints(*args) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (lùi duy nhất).** Với `tx ≠ ty` dương, `(tx, ty)` có đúng một điểm trước hợp lệ (toạ độ dương): `(tx - ty, ty)` nếu `tx > ty`, `(tx, ty - tx)` nếu `ty > tx`. Với `tx = ty` không có điểm trước. (Chứng minh ở mục 4.2.)

**Hệ quả.** Tập các điểm có thể đi tới `(tx, ty)` là **một dãy** `P₀ = (tx,ty), P₁, P₂, ...` (dãy lùi). `(sx, sy)` tới được đích ⇔ `(sx, sy)` xuất hiện trong dãy này.

**Bổ đề 2 (đơn điệu).** Dọc dãy lùi, mỗi toạ độ không tăng. Vậy khi `tx < sx` hoặc `ty < sy` thì `(sx, sy)` không thể xuất hiện về sau → dừng và trả `False` là đúng.

**Bổ đề 3 (nhảy bằng `%` không bỏ sót).** Xét `tx > ty > sy`. Các điểm tiếp theo trong dãy là `(tx - ty, ty), (tx - 2ty, ty), ...` cho tới khi `x < ty`, tức tới `(tx % ty, ty)`. Tất cả các điểm bị nhảy qua có `y = ty > sy`, nên **không** điểm nào là `(sx, sy)`. Vậy nhảy thẳng tới `(tx % ty, ty)` không bỏ sót.

(Lưu ý `tx % ty` có thể bằng 0 — khi đó `0 < sx` nên vòng lặp kết thúc và trả `False`, đúng vì `ty | tx` nghĩa là dãy lùi chạm `(ty, ty)` rồi dừng, mà `(ty, ty) ≠ (sx, sy)` do `ty > sy`.)

**Bổ đề 4 (nhánh `ty == sy`).** Khi `tx > ty = sy`, các điểm tiếp theo là `(tx - k·ty, sy)` cho `k = 1, 2, ...` (đến khi `x ≤ ty`). `(sx, sy)` có mặt ⇔ tồn tại `k ≥ 0` với `tx - k·ty = sx` ⇔ `(tx - sx) % ty == 0` (vì `tx ≥ sx`). Chú ý: với `sx ≤ ty` cũng không sao vì trong khi trừ, điểm `(sx, sy)` sẽ gặp trước khi `x` xuống dưới `sx`.

Kết hợp các bổ đề, thuật toán duyệt (dưới dạng nén) đúng dãy lùi và dừng đúng lúc.

## 11. Phân tích Time/Space

- **Cách 1 (BFS xuôi):** `O(tx · ty)` thời gian/bộ nhớ — chỉ dùng cho số nhỏ.
- **Cách 2 (trừ):** `O(max(tx, ty))` bước xấu nhất, `O(1)` bộ nhớ.
- **Cách 3 (`%` vòng lặp):** mỗi lần `%` giống một bước Euclid; sau 2 bước liên tiếp số lớn giảm ít nhất một nửa → **`O(log max(tx, ty))`**, `O(1)` bộ nhớ.
- **Cách 4 (đệ quy):** `O(log max)` thời gian, `O(log max)` ngăn xếp.

Đo số vòng lặp của cách 3 ở trường hợp "Fibonacci" (xấu nhất của Euclid):

In [17]:
def count_loops(sx, sy, tx, ty):
    loops = 0
    while tx >= sx and ty >= sy:
        loops += 1
        if tx == ty:
            break
        if tx > ty:
            if ty > sy: tx %= ty
            else: return loops
        else:
            if tx > sx: ty %= tx
            else: return loops
    return loops

fib = [1, 1]
while fib[-1] < 10**9:
    fib.append(fib[-1] + fib[-2])
print("Fibonacci lớn nhất <= 1e9:", fib[-3], fib[-2])
print("số vòng:", count_loops(1, 1, fib[-3], fib[-2]), "| log2(1e9) ≈ 30")
print("(1,1)->(1e9,1): số vòng =", count_loops(1, 1, 10**9, 1))

Fibonacci lớn nhất <= 1e9: 433494437 701408733
số vòng: 42 | log2(1e9) ≈ 30
(1,1)->(1e9,1): số vòng = 1


## 12. Bộ test

In [18]:
import random

methods = {
    "bfs": reach_bfs,
    "subtract": reach_subtract,
    "mod": reach_mod,
    "rec": reach_rec,
    "SolutionSubtract": lambda *a: SolutionSubtract().reachingPoints(*a),
    "SolutionMod": lambda *a: SolutionMod().reachingPoints(*a),
    "SolutionRec": lambda *a: SolutionRec().reachingPoints(*a),
    "Solution": lambda *a: Solution().reachingPoints(*a),
}
fast = ["mod", "rec", "SolutionMod", "SolutionRec", "Solution"]

# 12.1 Ví dụ + edge cases nhỏ
edge = [
    ((1, 1, 3, 5), True), ((1, 1, 2, 2), False), ((1, 1, 1, 1), True),
    ((2, 3, 2, 3), True),        # không cần bước nào
    ((2, 3, 3, 2), False),       # đổi chỗ không được
    ((1, 2, 1, 7), True),        # chỉ tăng y
    ((3, 3, 12, 9), True),       # (3,3)->(3,6)->(9,6)? 9,6 -> ... kiểm tra bằng bfs
    ((4, 6, 4, 6), True),
    ((1, 5, 6, 5), True),        # (1,5)->(6,5)
    ((5, 1, 5, 11), True),       # (5,1)->(5,6)->(5,11)
    ((3, 7, 3, 4), False),       # ty < sy ngay từ đầu
]
for name, f in methods.items():
    for args, _ in edge:
        assert f(*args) == reach_bfs(*args), (name, args)
for args, exp in EXAMPLES[:5]:
    for name, f in methods.items():
        assert f(*args) == exp, (name, args)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu với BFS
rnd = random.Random(780)
cnt_true = 0
for _ in range(2000):
    sx, sy = rnd.randint(1, 10), rnd.randint(1, 10)
    tx, ty = rnd.randint(1, 60), rnd.randint(1, 60)
    ref = reach_bfs(sx, sy, tx, ty)
    cnt_true += ref
    for name, f in methods.items():
        assert f(sx, sy, tx, ty) == ref, (name, sx, sy, tx, ty)
print("2000 bộ ngẫu nhiên khớp BFS; số True =", cnt_true)

# 12.3 Sinh đích bằng cách đi xuôi ngẫu nhiên (đảm bảo True), số lớn
for _ in range(500):
    sx, sy = rnd.randint(1, 10**4), rnd.randint(1, 10**4)
    x, y = sx, sy
    for _ in range(rnd.randint(0, 40)):
        if rnd.random() < 0.5:
            x += y
        else:
            y += x
        if max(x, y) > 10**9:
            break
    for name in fast:
        assert methods[name](sx, sy, x, y) is True, (name, sx, sy, x, y)
    # biến đổi nhẹ đích -> thường False, đối chiếu giữa các cách nhanh
    vals = {methods[name](sx, sy, x + 1, y) for name in fast}
    assert len(vals) == 1
print("Đích sinh xuôi ngẫu nhiên (số lớn): OK")

# 12.4 Số cực lớn
for name in fast:
    assert methods[name](1, 1, 10**9, 1) is True
    assert methods[name](1, 1, 10**9, 10**9) is False
    assert methods[name](35, 13, 455955547, 420098884) is False
print("Số lớn: OK")

Ví dụ + edge cases: OK


2000 bộ ngẫu nhiên khớp BFS; số True = 117


Đích sinh xuôi ngẫu nhiên (số lớn): OK
Số lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `%` máy móc, quên trường hợp một toạ độ đã bằng toạ độ xuất phát

**Code sai:**

In [19]:
def wrong_always_mod(sx, sy, tx, ty):
    while tx > sx and ty > sy:          # SAI: điều kiện chặt + luôn %
        if tx > ty:
            tx %= ty
        else:
            ty %= tx
    return tx == sx and ty == sy

for args in [(1, 1, 3, 5), (1, 1, 10**9, 1), (2, 3, 12, 29), (1, 5, 6, 5)]:
    print(args, "sai:", wrong_always_mod(*args), " đúng:", Solution().reachingPoints(*args))

(1, 1, 3, 5) sai: False  đúng: True
(1, 1, 1000000000, 1) sai: False  đúng: True
(2, 3, 12, 29) sai: False  đúng: True
(1, 5, 6, 5) sai: False  đúng: True


**Vì sao sai:** khi `ty == sy`, `tx %= ty` nhảy **quá** `sx` (ví dụ `(1,1)→(10⁹,1)`: `10⁹ % 1 = 0 ≠ 1`). Ngoài ra vòng `while` chặt dừng sớm mà không xử lý phần còn lại (`(1,1)→(3,5)` dừng ở `(1,2)`).

**Code đúng:** tách nhánh `ty == sy` → kiểm tra `(tx - sx) % ty == 0` (cách 3).

### Lỗi 2 — Đi ngược bằng phép trừ trên số lớn → TLE

Cách 2 đúng nhưng `(1, 1, 10⁹, 1)` cần `10⁹` vòng lặp. Luôn gộp phép trừ lặp thành `%`.

In [20]:
import time
t0 = time.perf_counter()
try:
    reach_subtract(1, 1, 10**9, 1, max_steps=2 * 10**5)
except RuntimeError:
    pass
el = time.perf_counter() - t0
print(f"2e5 bước mất {el*1000:.1f} ms -> 1e9 bước ước chừng {el * 5000:.0f} s")

2e5 bước mất 35.4 ms -> 1e9 bước ước chừng 177 s


**Vì sao sai:** độ phức tạp tuyến tính theo giá trị chứ không theo số chữ số.

**Code đúng:** `tx %= ty` (khi `ty > sy`) hoặc kiểm tra chia hết (khi `ty == sy`).

### Lỗi 3 — Quên trường hợp `tx == ty` → vòng lặp vô hạn hoặc chia cho 0

**Code sai:**

In [21]:
def wrong_equal(sx, sy, tx, ty, max_iter=100):
    it = 0
    while tx >= sx and ty >= sy and it < max_iter:
        it += 1
        if (tx, ty) == (sx, sy):
            return True
        if tx >= ty:                 # SAI: tx == ty rơi vào đây
            if ty > sy:
                tx %= ty             # tx == ty -> tx = 0
            else:
                return (tx - sx) % ty == 0
        else:
            if tx > sx:
                ty %= tx
            else:
                return (ty - sy) % tx == 0
    return tx == sx and ty == sy

print("(1,1)->(2,2) sai:", wrong_equal(1, 1, 2, 2), "  đúng:", Solution().reachingPoints(1, 1, 2, 2))
print("(2,1)->(2,2) sai:", wrong_equal(2, 1, 2, 2), "  đúng:", Solution().reachingPoints(2, 1, 2, 2))

(1,1)->(2,2) sai: False   đúng: False
(2,1)->(2,2) sai: False   đúng: False


**Vì sao sai:** khi `tx == ty`, code gộp vào nhánh `tx > ty` và làm `tx %= ty` → `tx = 0`. Với ràng buộc `sx ≥ 1`, vòng lặp tình cờ thoát vì `0 < sx` nên kết quả vẫn đúng — code đúng **nhờ may mắn**, không nhờ lập luận. Chỉ cần đề cho phép toạ độ `0` (một biến thể hay gặp), vòng sau sẽ tính `% 0` → **chia cho 0**. Viết tường minh `if tx == ty: break` vừa an toàn vừa thể hiện đúng Bổ đề "không lùi được khi `tx == ty`".

In [22]:
def wrong_equal_zero(sx, sy, tx, ty):
    """Giả sử đề cho phép sx = 0: code thiếu xử lý tx == ty sẽ chia cho 0."""
    try:
        return wrong_equal(sx, sy, tx, ty)
    except ZeroDivisionError as e:
        return f"ZeroDivisionError: {e}"

print(wrong_equal_zero(0, 1, 2, 2))

ZeroDivisionError: integer modulo by zero


**Code đúng:** kiểm tra `tx == ty` trước, `break` rồi so sánh với `(sx, sy)`.

### Lỗi 4 — Đi xuôi bằng đệ quy không cắt nhánh

Đệ quy `f(x, y) = f(x+y, y) or f(x, x+y)` mà không dừng khi vượt đích sẽ chạy vô hạn (tới khi tràn ngăn xếp). Luôn cắt nhánh `x > tx or y > ty` — và kể cả vậy vẫn quá chậm với số lớn.

## 14. Mẹo & nhận dạng pattern

- **"Đi ngược từ đích"**: khi phép biến đổi xuôi có nhiều nhánh nhưng phép ngược là **duy nhất** (thường do so sánh hai toạ độ), hãy đi ngược.
- **"Trừ lặp = chia lấy dư"**: mỗi khi thấy vòng lặp `while a > b: a -= b`, nghĩ ngay tới `a %= b` và **cẩn thận điều kiện dừng** (không được nhảy qua mục tiêu).
- **Dấu hiệu Euclid**: phép biến đổi `(x, y) → (x+y, y)` / `(x, x+y)` giữ nguyên `gcd(x, y)`. Điều kiện cần: `gcd(sx, sy) == gcd(tx, ty)` (nhưng không đủ).
- Độ phức tạp `O(log)` vì giống Euclid — trường hợp xấu nhất là các số Fibonacci liên tiếp.

In [23]:
from math import gcd
# Điều kiện cần nhưng không đủ: gcd bảo toàn
for args in [(1, 1, 3, 5), (2, 3, 12, 29), (2, 3, 11, 30), (1, 1, 2, 2)]:
    sx, sy, tx, ty = args
    print(args, "gcd đầu =", gcd(sx, sy), "gcd đích =", gcd(tx, ty), "| reach =", Solution().reachingPoints(*args))

(1, 1, 3, 5) gcd đầu = 1 gcd đích = 1 | reach = True
(2, 3, 12, 29) gcd đầu = 1 gcd đích = 1 | reach = True
(2, 3, 11, 30) gcd đầu = 1 gcd đích = 1 | reach = False
(1, 1, 2, 2) gcd đầu = 1 gcd đích = 2 | reach = False


## 15. Biến thể

1. **Đếm số bước tối thiểu** (= số bước duy nhất, vì đường lùi duy nhất): cộng `tx // ty` (hoặc `(tx - sx) // ty`) mỗi lần dùng `%`.
2. **Cho phép cả `(x - y, y)`...**: đồ thị không còn cây → cần lý thuyết số (gcd) để trả lời.
3. **Phép `(x, y) → (x + k·y, y)` với `k` cố định**: đi ngược vẫn duy nhất nhưng điều kiện chia hết đổi thành theo `k·y`.
4. **1342 / 2543 Check if Point Is Reachable**: các phép `(x, y-x)`, `(x-y, y)`, `(2x, y)`, `(x, 2y)` → đáp án: `gcd(tx, ty)` là luỹ thừa của 2.
5. **Đếm số điểm xuất phát `(a, b)` với `a, b ≤ N` tới được `(tx, ty)`**: chính là độ dài dãy lùi.

Minh hoạ biến thể 1 — đếm số bước:

In [24]:
def count_steps(sx, sy, tx, ty):
    """Số phép biến đổi từ (sx,sy) tới (tx,ty), hoặc -1 nếu không tới được."""
    steps = 0
    while tx >= sx and ty >= sy:
        if tx == sx and ty == sy:
            return steps
        if tx == ty:
            return -1
        if tx > ty:
            if ty > sy:
                steps += tx // ty
                tx %= ty
            else:
                if (tx - sx) % ty:
                    return -1
                return steps + (tx - sx) // ty
        else:
            if tx > sx:
                steps += ty // tx
                ty %= tx
            else:
                if (ty - sy) % tx:
                    return -1
                return steps + (ty - sy) // tx
    return -1

print(count_steps(1, 1, 3, 5), count_steps(2, 3, 12, 29), count_steps(1, 1, 10**9, 1), count_steps(1, 1, 2, 2))

3 5 999999999 -1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS xuôi | duyệt cây biến đổi, cắt nhánh | `O(tx·ty)` | `O(tx·ty)` | đối chiếu với số nhỏ |
| 2. Lùi bằng trừ | lùi duy nhất | `O(max(tx,ty))` | `O(1)` | hiểu bài; số nhỏ |
| 3. Lùi bằng `%` | Euclid + nhánh `== s` | `O(log max)` | `O(1)` | **lời giải chuẩn** |
| 4. Đệ quy nhảy `k` bước | trừ tối đa mà vẫn `≥ s` | `O(log max)` | `O(log max)` | cách viết thay thế, dễ chứng minh |

## 17. Bài tập

**Bài 1 (đoán output).** `reachingPoints(1, 2, 5, 3)`, `reachingPoints(1, 2, 3, 5)`, `reachingPoints(4, 1, 4, 13)` = ?

**Bài 2 (tính tay).** Đi ngược từ `(19, 7)` về phía `(1, 1)`: viết dãy lùi (dạng nén bằng `%`). Có tới `(5, 2)` không?

**Bài 3 (điền TODO).** Hoàn thiện `path_forward(sx, sy, tx, ty)` trả về **danh sách điểm** từ `(sx, sy)` tới `(tx, ty)` (hoặc `[]`), với `tx, ty ≤ 1000`.

**Bài 4 (tự viết lại).** Viết lại cách 3 sao cho **chỉ có một nhánh** bằng cách hoán đổi vai trò `x`/`y` (luôn đảm bảo `tx ≥ ty` bằng swap cả cặp đích và cặp xuất phát).

**Bài 5 (mini challenge).** Với `N = 30`, đếm có bao nhiêu điểm `(a, b)` với `1 ≤ a, b ≤ N` tới được `(N, N - 7)`... tổng quát: viết `count_sources(tx, ty)` trả về số điểm `(a, b)` (dương) tới được `(tx, ty)`.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def path_forward(sx, sy, tx, ty):
    # TODO: đi ngược bằng phép trừ, lưu các điểm, đảo ngược
    pass

# Bài 4 — TODO
def reach_one_branch(sx, sy, tx, ty):
    # TODO: nếu tx < ty: hoán đổi (sx,sy) và (tx,ty) để tx >= ty
    pass

# Bài 5 — TODO
def count_sources(tx, ty):
    # TODO: độ dài dãy lùi = tổng các thương trong Euclid (cẩn thận bước cuối)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: lùi từ `(5, 3)`: `tx > ty`.
- Mức 2: `(5,3) → (2,3) → (2,1)`: `ty = 1 < sy = 2`... nhưng ở `(2,3)` thì `ty > tx = 2 > sx = 1` → `ty %= 2 = 1 < 2` → `False`. `(3,5) → (3,2) → (1,2)` ✔ `True`.
- Mức 3: `(4,13)`: `tx == sx = 4` → `(13 - 1) % 4 == 0` → `True`.

**Bài 2.**
- Mức 1: `19 % 7 = 5`.
- Mức 2: `(19,7) → (5,7) → (5,2)`.
- Mức 3: có, `(5, 2)` nằm trên dãy lùi (dạng nén) — kiểm tra lại bằng `Solution`.

**Bài 3.**
- Mức 1: lùi từng bước bằng phép trừ (số nhỏ nên đủ nhanh).
- Mức 2: dừng khi bằng `(sx, sy)` hoặc thất bại.
- Mức 3: đảo danh sách để có thứ tự xuôi.

**Bài 4.**
- Mức 1: hoán đổi `x ↔ y` không làm thay đổi bài toán (phép biến đổi đối xứng).
- Mức 2: trong vòng lặp, sau mỗi `%` có thể phải hoán đổi lại.
- Mức 3: dùng vòng `while` với `if tx < ty: swap` ở đầu mỗi vòng.

**Bài 5.**
- Mức 1: mọi nguồn đều nằm trên dãy lùi.
- Mức 2: đếm số điểm của dãy lùi (dương) — gồm cả chính `(tx, ty)`.
- Mức 3: đếm bằng thương `tx // ty`, nhưng nếu chia hết thì bước cuối dừng ở `(ty, ty)` (không xuống `0`).

## 19. Đáp án tham khảo

In [26]:
S = Solution()
# Bài 1
assert S.reachingPoints(1, 2, 5, 3) is False
assert S.reachingPoints(1, 2, 3, 5) is True
assert S.reachingPoints(4, 1, 4, 13) is True

# Bài 2
assert backward_path(19, 7)[:3] == [(19, 7), (12, 7), (5, 7)]
assert S.reachingPoints(5, 2, 19, 7) is True

# Bài 3
def path_forward_ans(sx, sy, tx, ty):
    path = [(tx, ty)]
    while (tx, ty) != (sx, sy):
        if tx < sx or ty < sy or tx == ty:
            return []
        if tx > ty:
            tx -= ty
        else:
            ty -= tx
        path.append((tx, ty))
    return path[::-1]

p = path_forward_ans(1, 1, 3, 5)
assert p == [(1, 1), (1, 2), (3, 2), (3, 5)]
assert path_forward_ans(1, 1, 2, 2) == []
for _ in range(300):
    a, b, c, d = rnd.randint(1, 8), rnd.randint(1, 8), rnd.randint(1, 80), rnd.randint(1, 80)
    pp = path_forward_ans(a, b, c, d)
    assert (len(pp) > 0) == reach_bfs(a, b, c, d)
    for (x1, y1), (x2, y2) in zip(pp, pp[1:]):
        assert (x2, y2) in successors(x1, y1)

# Bài 4
def reach_one_branch_ans(sx, sy, tx, ty):
    while tx >= sx and ty >= sy:
        if tx < ty:
            sx, sy, tx, ty = sy, sx, ty, tx
        if tx == ty:
            break
        if ty > sy:
            tx %= ty
        else:
            return (tx - sx) % ty == 0
    return tx == sx and ty == sy

for _ in range(2000):
    a, b, c, d = rnd.randint(1, 10), rnd.randint(1, 10), rnd.randint(1, 60), rnd.randint(1, 60)
    assert reach_one_branch_ans(a, b, c, d) == reach_bfs(a, b, c, d), (a, b, c, d)

# Bài 5
def count_sources_ans(tx, ty):
    cnt = 1                       # chính (tx, ty)
    while tx != ty:
        if tx < ty:
            tx, ty = ty, tx
        q, r = divmod(tx, ty)
        if r == 0:
            cnt += q - 1          # xuống tới (ty, ty) rồi dừng
            break
        cnt += q
        tx = r
    return cnt

def count_sources_brute(tx, ty):
    return sum(reach_bfs(a, b, tx, ty) for a in range(1, tx + 1) for b in range(1, ty + 1))

for tx in range(1, 25):
    for ty in range(1, 25):
        assert count_sources_ans(tx, ty) == count_sources_brute(tx, ty), (tx, ty)
print("count_sources(30, 23) =", count_sources_ans(30, 23))
print("Tất cả đáp án đúng")

count_sources(30, 23) = 9
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đi ngược kiểu Euclid":**

```python
while tx >= sx and ty >= sy:
    if tx == ty: break
    if tx > ty:
        if ty > sy: tx %= ty
        else: return (tx - sx) % ty == 0
    else:
        if tx > sx: ty %= tx
        else: return (ty - sy) % tx == 0
return tx == sx and ty == sy
```

**Checklist trước khi submit:**

- [ ] Đi **ngược** (từ đích), không đi xuôi.
- [ ] Dùng `%` thay cho trừ lặp.
- [ ] Nhánh `ty == sy` (hoặc `tx == sx`): kiểm tra chia hết, **không** `%` tiếp.
- [ ] Xử lý `tx == ty` (không lùi được nữa).
- [ ] Trường hợp không cần bước nào: `(sx, sy) == (tx, ty)` → `True`.
- [ ] Điều kiện vòng lặp `tx >= sx and ty >= sy` (không dùng `>` chặt).